### Download the negative tsv file from Uniprot using the Request API

In [41]:
import requests
import pandas as pd

url = "https://rest.uniprot.org/uniprotkb/stream"  

# parameters and structure of the query
params = {
    "query": "(reviewed:true) AND (fragment:false) AND (taxonomy_id:2759) "
             "AND (length:[40 TO *]) AND "
             "((cc_scl_term_exp:SL-0091) OR (cc_scl_term_exp:SL-0188) OR "
             "(cc_scl_term_exp:SL-0173) OR (cc_scl_term_exp:SL-0209) OR "
             "(cc_scl_term_exp:SL-0204) OR (cc_scl_term_exp:SL-0039)) "
             "NOT (ft_signal:*)",          
    "fields": "accession,reviewed,id,length,ft_transmem,lineage,sequence",  
    "format": "tsv",   
}
r = requests.get(url, params=params, timeout=300)
r.raise_for_status()

# create the tsv file
with open("uniprot_results.tsv", "w") as f:
    f.write(r.text)

# convert the tsv file into a dataframe
df = pd.read_csv("uniprot_results.tsv", sep="\t")


In [43]:
# check the dataframe
print(df.info)
df.columns

<bound method DataFrame.info of             Entry  Reviewed   Entry Name  Length  \
0      A0A061ACU2  reviewed  PIEZ1_CAEEL    2442   
1      A0A067XGX8  reviewed  AROG2_PETHY     512   
2      A0A067XH53  reviewed  AROG1_PETHY     533   
3      A0A075D657  reviewed  PINMT_VINMI     322   
4      A0A075TRC0  reviewed   PATK_PENEN    1776   
...           ...       ...          ...     ...   
18974      Q9TKX7  reviewed  YCF81_NEPOL     138   
18975      Q9UT54  reviewed   YI95_SCHPO     125   
18976      Q9UTM1  reviewed   YIV1_SCHPO     112   
18977      Q9XPS5  reviewed  YCF70_WHEAT      42   
18978      Q9Y807  reviewed   YN92_SCHPO     263   

                                           Transmembrane  \
0      TRANSMEM 6..26; /note="Helical; Name=1"; /evid...   
1                                                    NaN   
2                                                    NaN   
3                                                    NaN   
4                                          

Index(['Entry', 'Reviewed', 'Entry Name', 'Length', 'Transmembrane',
       'Taxonomic lineage', 'Sequence'],
      dtype='str')

### Filter the entries of the negative set 

In [60]:
import re
import pandas as pd

# this function checks if the transmembrane helics startpoint starts within the first 90 residues
def tm_starts(s):
    if pd.isna(s):
        return []
    return [int(x) for x in re.findall(r'TRANSMEM\s+[<>?]?(\d+)', str(s))]

# theese two functions split the Taxonomic lineage field into kingdom and organism
def kingdom(s):
    parts = [p.strip() for p in str(s).split(',')]
    names = {re.sub(r'\s*\(.*\)$', '', p) for p in parts}
    if 'Metazoa' in names:         return 'Metazoa'
    if 'Fungi' in names:           return 'Fungi'
    if 'Viridiplantae' in names:   return 'Plants'
    return 'Other'

def organism(s):
    parts = [p.strip() for p in str(s).split(',')]
    sp = [p for p in parts if p.endswith('(species)')]
    last = sp[-1] if sp else parts[-1]
    return re.sub(r'\s*\(.*\)$', '', last)

# create the dataframe with the filtered data
filtered_df = pd.DataFrame({
    'accession':      df['Entry'],
    'organism':       df['Taxonomic lineage'].apply(organism),
    'kingdom':        df['Taxonomic lineage'].apply(kingdom),
    'length':         df['Length'].astype(int),
    'tm_in_first_90': df['Transmembrane'].apply(lambda s: any(p <= 90 for p in tm_starts(s))),
})

# convert the dataframe into a tsv file
filtered_df.to_csv('negative_data.tsv', sep='\t', index=False)

In [66]:
# check the new df
print(filtered_df.head(10))
print()
print(filtered_df['tm_in_first_90'].value_counts())

    accession                    organism  kingdom  length  tm_in_first_90
0  A0A061ACU2              Caenorhabditis  Metazoa    2442            True
1  A0A067XGX8                     Petunia   Plants     512           False
2  A0A067XH53                     Petunia   Plants     533           False
3  A0A075D657                       Vinca   Plants     322           False
4  A0A075TRC0                 Penicillium    Fungi    1776           False
5  A0A076FFM5                      Ocimum   Plants     523           False
6  A0A078CGE6                    Brassica   Plants    1299           False
7  A0A087X1C5                        Homo  Metazoa     515            True
8  A0A095C325  Cryptococcus deuterogattii    Fungi    1408           False
9  A0A096LP01                        Homo  Metazoa      95            True

tm_in_first_90
False    14970
True      4009
Name: count, dtype: int64


### Create filtered fasta file

In [69]:
df['Transmembrane'][5]

'TRANSMEM 466..486; /note="Helical"; /evidence="ECO:0000255"; TRANSMEM 489..509; /note="Helical"; /evidence="ECO:0000255"'

In [ ]:
with open("negative_SP_clean.fasta", "w") as f:
    for _, row in df.iterrows():
        f.write(f">{row['Entry']}\n{row['Sequence']}\n")   